# 00 · Inventaire des données

**Projet 4 : Analyse des données de vente et retours** · Sprint 0 · US02 et US04

**Objectif** : savoir exactement ce que contiennent nos données avant toute transformation.

| Étape | Question à laquelle on répond |
|---|---|
| 1. Fichiers | Quels fichiers de ventes avons-nous ? Manque-t-il des trimestres ? Y a-t-il des doublons ? |
| 2. Structure | Combien de lignes et de colonnes ? Quels types ? Les montants sont-ils convertibles en nombres ? |
| 3. Période | Chaque fichier couvre-t-il bien le trimestre annoncé dans son nom ? |
| 4. Unicité | Chaque vente (IdVente) est-elle unique ? |
| 5. Correspondance | Nos identifiants existent-ils dans les tables Contoso ? |
| 6. Dimensions | Les tables de dimension sont-elles saines (clés uniques, valeurs manquantes, liens entre elles) ? Quelle part est réellement utilisée ? |
| 7. Décalage de 12 ans | Les dates des promotions Contoso, décalées de +12 ans, coïncident-elles avec nos ventes ? |
| 8. Dictionnaire | Description et profil de chaque variable |
| 9. Périmètre | Que contiennent les fichiers fournis par rapport à la base Contoso ? Pourquoi travailler sur les ventes complètes ? |
| 10. Autres tables de faits | Que contiennent les charges, les objectifs, le stock et les ventes en ligne détaillées ? Leurs clés sont-elles uniques ? Se recoupent-elles avec nos ventes ? |
| 11. Synthèse | Les chiffres clés à reporter dans le rapport et la présentation |

**Lit dans** : `data/raw/contoso/Sales` (ventes complètes, source choisie dans `src/config.py`, variable `SOURCE_VENTES`), `data/raw/professeur` (fichiers du professeur, pour la comparaison de la section 9), `data/raw/contoso/Dim`, `data/raw/contoso/Plan`, `data/raw/contoso/Stock`
**Écrit dans** : `reports/figures`, `reports/livrables/L1_L2_phases_backlog/dictionnaire_donnees.xlsx`

**Fonctions partagées** : la lecture et la conversion viennent de `src/etl.py` (réutilisées par `01_etl`), le style des graphiques de `src/viz.py` (charte graphique du projet).

Ce notebook ne modifie aucune donnée : il observe et mesure. Les conversions sont faites en mémoire, uniquement pour pouvoir compter et contrôler. C'est `01_etl` qui enregistre les données converties dans `data/interim`.

**Changement de périmètre (6 octobre 2026)** : ce notebook a d'abord été exécuté sur les six fichiers fournis par le professeur. L'exploration de la base Contoso a montré que ces fichiers ne contiennent qu'une partie des ventes de nos boutiques. Avec l'accord du professeur, le projet travaille maintenant sur les **ventes complètes**. La section 9 documente cette découverte et justifie le choix.

**Année de référence et nouvelles tables (7 octobre 2026)** : l'analyse a montré qu'on ne peut pas expliquer la situation de 2021 en ne regardant que 2020 et 2021. La base contient une année de plus, 2019, et trois tables de faits que nous n'utilisions pas : le stock, les objectifs de vente et les ventes en ligne détaillées. Avec l'accord du professeur, elles sont extraites pour nos 56 boutiques. **Le périmètre de l'énoncé reste 2020 et 2021 ; 2019 sert d'année de référence**, pour dire d'où vient la situation. La section 10 fait l'inventaire de ces tables.

Toutes les données Contoso sont extraites par un seul script, `Extraction.py`, vers quatre dossiers : `Dim`, `Sales`, `Plan` et `Stock`.

## 0. Préparation

In [ ]:
import sys, re, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# 1. Retrouver la racine du projet (le dossier qui contient src/ et data/)
# ---------------------------------------------------------------------------
# Le notebook est dans notebooks/1_preparation/ : on remonte de dossier en dossier
# jusqu'à trouver celui qui contient à la fois src/ et data/.
# Si on arrive tout en haut du disque sans le trouver (racine == racine.parent),
# on arrête avec un message clair.
racine = Path.cwd()
while not ((racine / "src").is_dir() and (racine / "data").is_dir()):
    if racine == racine.parent:
        raise FileNotFoundError("Racine du projet introuvable : lancer ce notebook depuis le dossier Projet ADD ou un de ses sous-dossiers.")
    racine = racine.parent

# Ajouter la racine aux dossiers où Python cherche les modules,
# pour que "from src import ..." fonctionne depuis n'importe quel sous-dossier
sys.path.insert(0, str(racine))

# ---------------------------------------------------------------------------
# 2. Modules du projet
# ---------------------------------------------------------------------------
#   config -> chemins des dossiers et paramètres (décalage de 12 ans, noms de colonnes)
#   etl    -> fonctions de lecture et de conversion, partagées avec 01_etl
#   viz    -> charte graphique : couleurs, police, format des montants
from src import config, etl, viz

# Créer les dossiers de sortie s'ils n'existent pas encore (reports/figures, etc.)
config.creer_dossiers()

# Appliquer la charte graphique à tous les graphiques de ce notebook
viz.appliquer_style()

# ---------------------------------------------------------------------------
# 3. Affichage des tableaux
# ---------------------------------------------------------------------------
# Montrer jusqu'à 50 colonnes et élargir l'affichage pour éviter les coupures
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
print("Racine du projet :", config.ROOT)

## 1. Fichiers de ventes disponibles

In [ ]:
# ---------------------------------------------------------------------------
# 1. Lister les fichiers de ventes
# ---------------------------------------------------------------------------
# glob() cherche dans le dossier des ventes tous les fichiers dont le nom
# correspond au motif défini dans config (ex. "*.xlsx").
# sorted() garantit le même ordre à chaque exécution.
fichiers = sorted(config.SALES.glob(config.MOTIF_FICHIERS_VENTES))


# ---------------------------------------------------------------------------
# 2. Empreinte du contenu d'un fichier
# ---------------------------------------------------------------------------
def empreinte(chemin):
    """Renvoie la signature MD5 (32 caractères) du contenu du fichier.

    Deux fichiers au contenu strictement identique ont la même empreinte,
    même si leurs noms diffèrent. Le moindre octet différent donne une
    empreinte totalement différente.
    """
    return hashlib.md5(chemin.read_bytes()).hexdigest()


# ---------------------------------------------------------------------------
# 3. Construire une fiche descriptive par fichier
# ---------------------------------------------------------------------------
lignes = []
for f in fichiers:
    # Extraire l'année et le trimestre à partir du NOM du fichier (sans extension).
    #   (\d{4})  -> 4 chiffres, groupe 1 : l'année
    #   \s*      -> zéro ou plusieurs espaces
    #   T        -> la lettre T
    #   (\d)     -> 1 chiffre, groupe 2 : le trimestre
    # Exemples reconnus : "2020 T3", "2020T3", "2020  T3".
    # m vaut None si le nom ne respecte pas ce format.
    m = re.search(r"(\d{4})\s*T(\d)", f.stem)

    lignes.append({
        "fichier": f.name,                                  # nom complet avec extension
        "annee_nom": int(m.group(1)) if m else None,        # année annoncée par le nom (None si non reconnu)
        "trimestre_nom": int(m.group(2)) if m else None,    # trimestre annoncé par le nom (None si non reconnu)
        "taille_Mo": round(f.stat().st_size / 1e6, 2),      # taille : octets -> mégaoctets, 2 décimales
        "empreinte": empreinte(f),                          # signature du contenu
    })

# Liste de dictionnaires -> tableau : une ligne par fichier, une colonne par clé
inventaire_fichiers = pd.DataFrame(lignes)

# Affichage sans la colonne "empreinte" (longue et illisible).
# drop() ne la retire que pour l'affichage : elle reste dans inventaire_fichiers.
display(inventaire_fichiers.drop(columns="empreinte"))


# ---------------------------------------------------------------------------
# 4. Comparer les trimestres attendus et les trimestres présents
# ---------------------------------------------------------------------------
# Toutes les combinaisons (année, trimestre) des années étudiées : l'année de référence (2019)
# puis les deux années de l'énoncé, ex. (2019, 1), (2019, 2), ... (2021, 4)
attendus = {(a, t) for a in config.ANNEES_ETUDE for t in config.TRIMESTRES_ATTENDUS}

# Couples (année, trimestre) réellement annoncés par les noms de fichiers.
# zip() associe les deux colonnes ligne par ligne, set() en fait un ensemble.
presents = set(zip(inventaire_fichiers.annee_nom, inventaire_fichiers.trimestre_nom))

# Différence d'ensembles : ce qui est attendu mais absent
manquants_nom = sorted(attendus - presents)


# ---------------------------------------------------------------------------
# 5. Détecter les doublons
# ---------------------------------------------------------------------------
# keep=False marque TOUTES les occurrences d'un doublon (pas seulement la 2e),
# pour voir l'ensemble des fichiers concernés.

# a) Plusieurs fichiers dont le nom annonce le même trimestre
#    (ex. "Ventes 2020 T3.xlsx" et "Ventes 2020 T3 (1).xlsx").
#    Attention : les fichiers au nom non reconnu (None, None) ressortent aussi ici.
noms_en_double = inventaire_fichiers[
    inventaire_fichiers.duplicated(["annee_nom", "trimestre_nom"], keep=False)
]

# b) Plusieurs fichiers au contenu strictement identique, quel que soit leur nom
contenus_identiques = inventaire_fichiers[
    inventaire_fichiers.duplicated("empreinte", keep=False)
]


# ---------------------------------------------------------------------------
# 6. Résumé des contrôles
# ---------------------------------------------------------------------------
# Astuce "or 'aucun'" : une liste vide vaut False, donc [] or "aucun" -> "aucun".
print(f"\n{len(fichiers)} fichiers trouvés, {len(attendus)} trimestres attendus")
print("Trimestres absents (d'après les noms) :", [f"{a} T{t}" for a, t in manquants_nom] or "aucun")
print("Noms désignant le même trimestre :", noms_en_double.fichier.tolist() or "aucun")
print("Fichiers au contenu identique :", contenus_identiques.fichier.tolist() or "aucun")

## 2. Chargement et structure

In [ ]:
# ---------------------------------------------------------------------------
# 1. Lecture robuste d'un CSV : fonction etl.lire_csv
# ---------------------------------------------------------------------------
# La fonction est maintenant dans src/etl.py, car 01_etl l'utilise aussi.
# Elle essaie les encodages du plus strict au plus permissif :
#   utf-8-sig -> UTF-8, en ignorant le BOM éventuel (marqueur invisible
#                ajouté par Excel au début du fichier)
#   cp1252    -> encodage Windows d'Europe de l'Ouest (accents français)
#   latin-1   -> accepte n'importe quel octet, donc n'échoue jamais :
#                c'est le dernier recours
# Elle lit toutes les colonnes comme du texte (dtype=str) : rien n'est transformé
# à la lecture, la conversion des types se fait ensuite, de façon contrôlée.
# Elle nettoie aussi les noms de colonnes (espaces en début et en fin retirés).
# Elle renvoie un couple (DataFrame, encodage utilisé).


# ---------------------------------------------------------------------------
# 2. Charger tous les fichiers et noter leur structure
# ---------------------------------------------------------------------------
brut = {}        # dictionnaire : nom du fichier -> DataFrame brut
structure = []   # une fiche descriptive par fichier

for f in fichiers:
    # Lecture du fichier : on récupère les données et l'encodage qui a fonctionné
    df, enc = etl.lire_csv(f)

    # Conserver le DataFrame brut, accessible par le nom du fichier
    brut[f.name] = df

    structure.append({
        "fichier": f.name,
        "encodage": enc,            # encodage retenu pour ce fichier
        "lignes": len(df),          # nombre de lignes (hors en-tête)
        "colonnes": df.shape[1],    # nombre de colonnes (shape = (lignes, colonnes))
    })

# Liste de dictionnaires -> tableau récapitulatif, une ligne par fichier
structure = pd.DataFrame(structure)
display(structure)


# ---------------------------------------------------------------------------
# 3. Vérifier que tous les fichiers ont les mêmes colonnes
# ---------------------------------------------------------------------------
# Colonnes du premier fichier, prises comme référence.
#   brut.values()  -> les DataFrames du dictionnaire
#   iter(...)      -> crée un itérateur dessus
#   next(...)      -> en prend le premier élément
reference = list(next(iter(brut.values())).columns)

for nom, df in brut.items():
    # Comparaison de listes : mêmes noms ET même ordre
    if list(df.columns) != reference:
        # ^ = différence symétrique : colonnes présentes dans l'un des deux
        # fichiers mais pas dans l'autre (en trop ou manquantes).
        # Si l'ensemble affiché est vide, les noms sont les mêmes
        # et seul l'ordre des colonnes diffère.
        print(f"Colonnes différentes dans {nom} :", set(df.columns) ^ set(reference))


# ---------------------------------------------------------------------------
# 4. Résumé
# ---------------------------------------------------------------------------
print("Colonnes :", reference)

# Total des lignes de tous les fichiers.
# Le format ":," met une virgule comme séparateur de milliers (1,234,567),
# puis replace() la remplace par une espace, à la française (1 234 567).
print(f"Total : {structure.lignes.sum():,} lignes".replace(",", " "))

In [ ]:
# ---------------------------------------------------------------------------
# 1. Conversion des types : fonction etl.typer_ventes
# ---------------------------------------------------------------------------
# Les fonctions de conversion sont dans src/etl.py, car 01_etl applique exactement
# les mêmes pour produire les données de data/interim. Ici, on s'en sert uniquement
# pour CONTRÔLER que tout est convertible : rien n'est enregistré.
#
# etl.typer_ventes(df) applique à un fichier brut :
#   - etl.convertir_date    : essaie plusieurs formats de date (mois/jour, jour/mois, ISO)
#                             et garde celui qui convertit le plus de valeurs
#   - etl.convertir_montant : '$1,062.60 ' -> 1062.60 ; '($5.00)' -> -5.00 ; vide -> NaN
#   - pd.to_numeric         : quantités et identifiants, après retrait des espaces
# Elle renvoie trois choses :
#   v      -> une COPIE du fichier avec les bons types (le brut reste intact)
#   fmt    -> le format de date retenu pour ce fichier
#   echecs -> pour chaque colonne en échec, le nombre de valeurs présentes dans le brut
#             mais non convertibles (dictionnaire vide si tout est converti)


# ---------------------------------------------------------------------------
# 2. Appliquer les conversions à chaque fichier
# ---------------------------------------------------------------------------
ventes_par_fichier = {}   # nom du fichier -> DataFrame avec les bons types
conversion = []           # une fiche de contrôle par fichier

for nom, df in brut.items():
    v, fmt, echecs = etl.typer_ventes(df)
    ventes_par_fichier[nom] = v

    # Fiche du fichier : format de date retenu + une colonne "échecs X"
    # uniquement pour les colonnes ayant au moins un échec.
    # ** déplie le dictionnaire pour en ajouter les clés à la fiche.
    conversion.append({
        "fichier": nom,
        "format_date": fmt,
        **{f"échecs {k}": n for k, n in echecs.items()},
    })


# ---------------------------------------------------------------------------
# 3. Tableau de contrôle
# ---------------------------------------------------------------------------
# Les fichiers n'ont pas tous les mêmes colonnes "échecs" : pandas met NaN
# là où un fichier n'a pas d'échec pour une colonne, et fillna(0) le remplace par 0.
display(pd.DataFrame(conversion).fillna(0))
print("Une colonne « échecs » apparaît seulement si des valeurs n'ont pas pu être converties.")

## 3. Période couverte par chaque fichier

In [ ]:
# ---------------------------------------------------------------------------
# 1. Comparer la période réelle de chaque fichier à celle annoncée par son nom
# ---------------------------------------------------------------------------
periode = []
for nom, v in ventes_par_fichier.items():
    # Retrouver la fiche du fichier dans l'inventaire (année et trimestre du NOM).
    # set_index("fichier") permet de chercher une ligne par le nom du fichier.
    ligne = inventaire_fichiers.set_index("fichier").loc[nom]

    # Colonne des dates converties de ce fichier
    d = v[config.COL_DATE]

    # Trimestres réellement présents dans les DONNÉES :
    #   d.dt.year / d.dt.quarter -> année et trimestre de chaque date
    #   zip + set                -> couples (année, trimestre) distincts
    #   - {(nan, nan)}           -> on retire le couple vide dû aux dates illisibles
    trimestres_reels = sorted(set(zip(d.dt.year, d.dt.quarter)) - {(np.nan, np.nan)})

    # Trimestre annoncé par le nom du fichier
    annonce = (ligne.annee_nom, ligne.trimestre_nom)

    periode.append({
        "fichier": nom,
        "date_min": d.min().date() if d.notna().any() else None,   # première date du fichier
        "date_max": d.max().date() if d.notna().any() else None,   # dernière date du fichier
        "trimestres_réels": ", ".join(f"{int(a)} T{int(t)}" for a, t in trimestres_reels),
        # Conforme si le fichier contient UN SEUL trimestre et que c'est celui du nom
        "conforme_au_nom": trimestres_reels == [annonce],
    })

periode = pd.DataFrame(periode)
display(periode)


# ---------------------------------------------------------------------------
# 2. Réunir tous les fichiers en une seule table (en mémoire seulement)
# ---------------------------------------------------------------------------
# concat empile les fichiers les uns sous les autres.
#   keys=...            -> étiquette chaque bloc avec le nom de son fichier
#   names=[...]         -> nomme les deux niveaux de l'index (fichier, numéro de ligne)
#   reset_index(level=0)-> transforme le niveau "fichier" en colonne ordinaire
ventes = pd.concat(
    ventes_par_fichier.values(),
    keys=ventes_par_fichier.keys(),
    names=["fichier", "ligne"],
).reset_index(level=0)


# ---------------------------------------------------------------------------
# 3. Trimestres attendus mais absents des DONNÉES (et non plus seulement des noms)
# ---------------------------------------------------------------------------
reels = set(zip(ventes[config.COL_DATE].dt.year, ventes[config.COL_DATE].dt.quarter))
manquants_reels = sorted(attendus - {(int(a), int(t)) for a, t in reels if pd.notna(a)})
print("Trimestres réellement absents des données :", [f"{a} T{t}" for a, t in manquants_reels] or "aucun")

In [ ]:
# ---------------------------------------------------------------------------
# Visualisation : chiffre d'affaires par mois, avec la charte graphique (src/viz.py)
# ---------------------------------------------------------------------------

# 1. Agréger par mois
#    dt.to_period("M") ramène chaque date à son mois (2020-03-25 -> 2020-03)
mois = ventes.groupby(ventes[config.COL_DATE].dt.to_period("M")).agg(
    nb_ventes=(config.COL_VENTE, "size"),       # nombre de ventes du mois
    ca=("Montant des ventes", "sum"),           # chiffre d'affaires du mois
)

# 2. Construire le calendrier COMPLET de la période étudiée (janv. 2019 à déc. 2021)
#    puis y replacer nos mois : ceux qui n'ont aucune vente apparaissent avec 0.
#    Sans cette étape, les mois absents disparaîtraient simplement du graphique.
calendrier = pd.period_range(
    f"{min(config.ANNEES_ETUDE)}-01", f"{max(config.ANNEES_ETUDE)}-12", freq="M")
mois = mois.reindex(calendrier, fill_value=0)
absents = mois.nb_ventes == 0                   # True pour les mois sans aucune vente

# 3. Tracer
fig, ax = plt.subplots(figsize=(13, 4.5))
positions = np.arange(len(mois))                # 0, 1, 2... : position de chaque mois sur l'axe

# Barres : couleur principale de la charte, montants en dollars (viz.format_dollars
# affichera "30,5 M$" sur l'axe)
# L'année de référence est tracée en gris, les deux années de l'énoncé dans la couleur principale
couleurs = np.where(mois.index.year == config.ANNEE_REFERENCE, viz.GRIS, viz.PRINCIPALE)
ax.bar(positions, mois.ca.astype(float), color=couleurs, width=0.8)

# Mois absents : une bande de fond sur toute la hauteur, pour qu'on les voie
# même si leur barre est à zéro (une barre de hauteur 0 est invisible)
for i in positions[absents.values]:
    ax.axvspan(i - 0.5, i + 0.5, color=viz.CLAIR, zorder=0)

# Étiqueter chaque groupe de mois absents consécutifs (ex. "2020 T4 absent")
for (a, t) in manquants_reels:
    # positions des mois de ce trimestre : (t-1)*3 mois après le début de l'année a
    debut = (a - min(config.ANNEES_ETUDE)) * 12 + (t - 1) * 3
    ax.text(debut + 1, mois.ca.astype(float).max() * 0.5, f"{a} T{t}\nabsent",
            ha="center", va="center", color=viz.ALERTE, fontsize=11, fontweight="bold")

# Axe horizontal : une étiquette par mois, écrite verticalement
ax.set_xticks(positions)
ax.set_xticklabels(mois.index.astype(str), rotation=90)
ax.set_xlim(-0.6, len(mois) - 0.4)

viz.format_dollars(ax, axe="y")
viz.titre(ax, "Chiffre d'affaires mensuel",
          f"Ventes complètes de nos 56 boutiques · en gris : {config.ANNEE_REFERENCE}, année de référence · en vert : années de l'énoncé")

# 4. Enregistrer dans reports/figures/00_ca_mensuel.png (résolution fixée par la charte)
chemin_figure = viz.enregistrer(fig, "00_ca_mensuel")
plt.show()
print("Figure enregistrée :", chemin_figure)

## 4. Unicité des ventes

In [ ]:
# ---------------------------------------------------------------------------
# Chaque vente doit apparaître une seule fois
# ---------------------------------------------------------------------------

# a) IdVente présents plusieurs fois.
#    keep=False marque TOUTES les occurrences d'un doublon (pas seulement la 2e).
doublons_total = ventes[config.COL_VENTE].duplicated(keep=False)

# b) Parmi ces doublons, lesquels viennent de fichiers DIFFÉRENTS ?
#    Pour chaque IdVente en double, on compte dans combien de fichiers il apparaît.
#    Plus d'un fichier = la même vente a été livrée dans deux trimestres.
entre_fichiers = ventes[doublons_total].groupby(config.COL_VENTE).fichier.nunique()

# c) Lignes entièrement identiques (toutes les colonnes), hors colonne "fichier"
lignes_identiques = ventes.drop(columns="fichier").duplicated().sum()

# Le format ":," met une virgule comme séparateur de milliers,
# remplacée par une espace à la française.
print(f"IdVente présents plusieurs fois : {doublons_total.sum():,} lignes".replace(",", " "))
print(f"dont IdVente présents dans plusieurs fichiers : {(entre_fichiers > 1).sum():,}".replace(",", " "))
print(f"Lignes entièrement identiques : {lignes_identiques:,}".replace(",", " "))

## 5. Correspondance des clés avec les tables Contoso

In [ ]:
# ---------------------------------------------------------------------------
# 1. Charger les tables de dimension
# ---------------------------------------------------------------------------
dims = {}             # nom de la table -> DataFrame
encodages_dims = {}   # nom de la table -> encodage qui a fonctionné

# Les dimensions des ventes, celles du plan stratégique, et les deux tables examinées pour le diagnostic
for t in config.TABLES_DIMENSION + config.TABLES_PLAN + config.TABLES_COMPLEMENT:
    chemin = config.DIM / f"{t}.csv"
    if chemin.exists():
        # etl.lire_dimension essaie les mêmes encodages que pour les ventes
        # et renvoie (DataFrame, encodage retenu)
        dims[t], encodages_dims[t] = etl.lire_dimension(t)
    else:
        # Table attendue mais absente du dossier : on le signale sans arrêter le notebook
        print(f"Table absente : {chemin}")


# ---------------------------------------------------------------------------
# 2. Nos clés existent-elles dans les tables Contoso ?
# ---------------------------------------------------------------------------
# config.CORRESPONDANCE_CLES associe chaque clé de nos ventes à sa table et sa colonne :
#   IdBoutique  -> DimStore.StoreKey
#   IdProduit   -> DimProduct.ProductKey
#   IdPromotion -> DimPromotion.PromotionKey
correspondance = []
for col, (table, cle) in config.CORRESPONDANCE_CLES.items():
    if table not in dims:
        continue                                   # table absente : rien à comparer

    connues = set(dims[table][cle])                # toutes les clés existantes dans Contoso
    presentes = ventes[col].isin(connues)          # True pour chaque vente dont la clé existe
    distinctes = set(ventes[col].dropna().unique())  # clés distinctes utilisées par nos ventes

    correspondance.append({
        "clé (nos ventes)": col,
        "table Contoso": f"{table}.{cle}",
        "valeurs distinctes": len(distinctes),
        # différence d'ensembles : clés utilisées par nos ventes mais inconnues de Contoso
        "distinctes introuvables": len(distinctes - connues),
        # mean() d'une colonne True/False = proportion de True
        "% des lignes retrouvées": round(100 * presentes.mean(), 2),
    })


# ---------------------------------------------------------------------------
# 3. Canal et devise : IdVente = FactSales.SalesKey
# ---------------------------------------------------------------------------
# Nos fichiers n'ont ni canal ni devise. Extraction.py a extrait de FactSales, pour chaque
# vente, sa clé (SalesKey), son canal (channelKey) et sa devise (CurrencyKey), dans le dossier des ventes.
fichier_cles = config.SALES_CONTOSO / config.FICHIER_CLES_CANAL_DEVISE
cles_utiles = None     # clés limitées à nos ventes (réutilisées à la section 6)

if fichier_cles.exists():
    cles = pd.read_csv(fichier_cles)
    presentes = ventes[config.COL_VENTE].isin(cles.SalesKey)
    correspondance.append({
        "clé (nos ventes)": config.COL_VENTE,
        "table Contoso": "FactSales.SalesKey",
        "valeurs distinctes": ventes[config.COL_VENTE].nunique(),
        "distinctes introuvables": int((~presentes).sum()),
        "% des lignes retrouvées": round(100 * presentes.mean(), 2),
    })

    # Ne garder que les clés de NOS ventes, puis compter les devises et les canaux.
    # value_counts() compte les lignes par valeur ; to_dict() l'affiche sur une ligne.
    cles_utiles = cles[cles.SalesKey.isin(ventes[config.COL_VENTE])]
    print("Devises utilisées par nos ventes :", cles_utiles.CurrencyKey.value_counts().to_dict())
    print("Canaux utilisés par nos ventes :", cles_utiles.channelKey.value_counts().to_dict())
else:
    print("Fichier des clés canal/devise absent : lancer Extraction.py")

correspondance = pd.DataFrame(correspondance)
display(correspondance)

## 6. Inventaire des tables de dimension

Les ventes ne sont pas les seules données du projet : les tables de dimension doivent être contrôlées elles aussi.
Pour chaque table, on vérifie :

- **l'encodage** et la **taille** (lignes, colonnes) ;
- **l'unicité de la clé primaire** : deux boutiques ne doivent pas partager le même StoreKey ;
- **les valeurs manquantes** : nombre de colonnes concernées et colonne la plus touchée ;
- **la période des colonnes de dates** : elles sont encore aux dates de Contoso (2007-2009 pour nos trois années), avant le réalignement de +12 ans ;
- **les liens entre dimensions** : boutiques sans géographie, produits sans sous-catégorie ;
- **la part réellement utilisée** par nos ventes.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Profil de chaque table de dimension
# ---------------------------------------------------------------------------
# Colonnes techniques de Contoso (suivi des chargements) : ce ne sont pas des dates métier,
# on les exclut de l'analyse des périodes.
COLONNES_TECHNIQUES = {"LoadDate", "UpdateDate", "ETLLoadID"}

profil_dims = []
for t, d in dims.items():
    # Dans Contoso, la clé primaire est toujours la première colonne de la table
    cle = d.columns[0]

    # Valeurs manquantes par colonne, en ne gardant que les colonnes concernées
    manquants = d.isna().sum()
    manquants = manquants[manquants > 0]

    # Colonnes de dates métier : nom terminé par "Date" (StartDate, OpenDate...),
    # hors colonnes techniques. DimDate est exclue : toutes ses colonnes décrivent des dates.
    cols_dates = [c for c in d.columns
                  if c.endswith("Date") and c not in COLONNES_TECHNIQUES and t != "DimDate"]
    if cols_dates:
        # Convertir ces colonnes en dates (errors="coerce" : l'illisible devient NaT),
        # puis prendre la plus ancienne et la plus récente, toutes colonnes confondues
        dates = d[cols_dates].apply(pd.to_datetime, errors="coerce")
        periode_dates = f"{dates.min().min().date()} au {dates.max().max().date()}"
    else:
        periode_dates = ""

    profil_dims.append({
        "table": t,
        "encodage": encodages_dims[t],
        "lignes": len(d),
        "colonnes": d.shape[1],
        "clé primaire": cle,
        # is_unique : True si aucune valeur de la clé n'est répétée
        "clé unique": bool(d[cle].is_unique),
        "clés manquantes": int(d[cle].isna().sum()),
        "colonnes avec manquants": len(manquants),
        # idxmax() donne la colonne ayant le plus de valeurs manquantes
        "colonne la plus incomplète": (f"{manquants.idxmax()} ({manquants.max()})" if len(manquants) else ""),
        "colonnes de dates": ", ".join(cols_dates),
        "période des dates": periode_dates,
    })

profil_dims = pd.DataFrame(profil_dims)
display(profil_dims)


# ---------------------------------------------------------------------------
# 2. Liens entre dimensions (doubles jointures du modèle en flocon)
# ---------------------------------------------------------------------------
# Chaque ligne : (table enfant, clé étrangère, table parent, clé du parent).
# Ce sont les relations déclarées dans la base (sys.foreign_keys).
RELATIONS = [
    ("DimStore", "GeographyKey", "DimGeography", "GeographyKey"),
    ("DimProduct", "ProductSubcategoryKey", "DimProductSubcategory", "ProductSubcategoryKey"),
    ("DimProductSubcategory", "ProductCategoryKey", "DimProductCategory", "ProductCategoryKey"),
]

liens = []
for enfant, cle_e, parent, cle_p in RELATIONS:
    if enfant not in dims or parent not in dims:
        continue
    valeurs = dims[enfant][cle_e]
    # Orphelin : ligne de l'enfant dont la clé étrangère n'existe pas chez le parent
    # (une boutique rattachée à une géographie inconnue, par exemple)
    orphelins = ~valeurs.isin(dims[parent][cle_p]) & valeurs.notna()
    liens.append({
        "relation": f"{enfant}.{cle_e} → {parent}.{cle_p}",
        "lignes": len(valeurs),
        "clés étrangères vides": int(valeurs.isna().sum()),
        "orphelins": int(orphelins.sum()),
    })

liens = pd.DataFrame(liens)
display(liens)


# ---------------------------------------------------------------------------
# 3. Part de chaque dimension réellement utilisée par nos ventes
# ---------------------------------------------------------------------------
# Nos ventes ne sont qu'un extrait : elles n'utilisent qu'une partie des boutiques,
# des produits et des promotions de Contoso.
utilisation = []
for col, (table, cle) in config.CORRESPONDANCE_CLES.items():
    if table not in dims:
        continue
    utilises = ventes[col].nunique()          # membres distincts présents dans nos ventes
    total = dims[table][cle].nunique()        # membres existants dans la dimension
    utilisation.append({
        "dimension": table,
        "membres dans Contoso": total,
        "membres utilisés par nos ventes": utilises,
        "% utilisé": round(100 * utilises / total, 1),
    })

# Canal : connu seulement si le fichier des clés a été trouvé (section 5)
if cles_utiles is not None and "DimChannel" in dims:
    utilises = cles_utiles.channelKey.nunique()
    total = dims["DimChannel"].iloc[:, 0].nunique()
    utilisation.append({"dimension": "DimChannel", "membres dans Contoso": total,
                        "membres utilisés par nos ventes": utilises,
                        "% utilisé": round(100 * utilises / total, 1)})

utilisation = pd.DataFrame(utilisation)
display(utilisation)

## 7. Vérification du décalage de 12 ans

Si nos dates sont bien celles de Contoso décalées de 12 ans, alors les ventes portant une promotion
doivent tomber **dans la période de cette promotion**, une fois ses dates décalées de +12 ans.
On compare plusieurs décalages : celui qui donne le meilleur taux est le bon.
La promotion 1 (« No Discount ») est exclue car elle couvre toute la période.

In [ ]:
if "DimPromotion" in dims:
    # -----------------------------------------------------------------------
    # 1. Préparer les promotions : clé, nom, date de début et date de fin
    # -----------------------------------------------------------------------
    promo = dims["DimPromotion"][["PromotionKey", "PromotionName", "StartDate", "EndDate"]].copy()
    promo["StartDate"] = pd.to_datetime(promo.StartDate)
    promo["EndDate"] = pd.to_datetime(promo.EndDate)

    # -----------------------------------------------------------------------
    # 2. Rattacher à chaque vente en promotion les dates de sa promotion
    # -----------------------------------------------------------------------
    # On écarte la promotion 1 (aucune remise), puis on joint sur la clé de promotion.
    # how="inner" : on ne garde que les ventes dont la promotion existe dans DimPromotion.
    avec_promo = ventes[ventes[config.COL_PROMOTION] != 1].merge(
        promo, left_on=config.COL_PROMOTION, right_on="PromotionKey", how="inner")

    # -----------------------------------------------------------------------
    # 3. Tester plusieurs décalages, de 10 à 14 ans
    # -----------------------------------------------------------------------
    test_decalage = []
    for ans in range(10, 15):
        # Période de la promotion, décalée de "ans" années.
        # On ajoute un jour à la fin pour inclure toute la dernière journée.
        debut = avec_promo.StartDate + pd.DateOffset(years=ans)
        fin = avec_promo.EndDate + pd.DateOffset(years=ans) + pd.Timedelta(days=1)

        # La vente tombe-t-elle dans cette période ?
        # inclusive="left" : début inclus, fin exclue (début <= date < fin)
        dans = avec_promo[config.COL_DATE].between(debut, fin, inclusive="left")

        test_decalage.append({
            "décalage (ans)": ans,
            "% ventes dans la période de leur promotion": round(100 * dans.mean(), 2),
        })

    test_decalage = pd.DataFrame(test_decalage)
    display(test_decalage)

    # -----------------------------------------------------------------------
    # 4. Conclusion : le décalage qui donne le taux le plus élevé
    # -----------------------------------------------------------------------
    # iloc[:, 1] = la colonne des pourcentages ; idxmax() = la ligne du maximum
    meilleur = test_decalage.loc[test_decalage.iloc[:, 1].idxmax(), "décalage (ans)"]
    print(f"Décalage le plus cohérent : {meilleur} ans (valeur retenue dans config : {config.DECALAGE_ANNEES})")

## 8. Dictionnaire de données

In [ ]:
# ---------------------------------------------------------------------------
# 1. Description de chaque variable des ventes
# ---------------------------------------------------------------------------
# Pour chaque colonne : (description, unité ou rôle, remarque).
# Les descriptions viennent de l'énoncé, les remarques de nos vérifications.
descriptions = {
    "IdDate": ("Date et heure de la transaction", "date", "Décalée de +12 ans par rapport à Contoso (DateKey)"),
    "IdVente": ("Identifiant unique de la vente", "identifiant", "= FactSales.SalesKey"),
    "IdBoutique": ("Identifiant de la boutique ou du point de vente", "clé étrangère", "→ DimStore.StoreKey"),
    "IdProduit": ("Identifiant du produit vendu", "clé étrangère", "→ DimProduct.ProductKey"),
    "IdPromotion": ("Identifiant de la promotion appliquée", "clé étrangère", "→ DimPromotion.PromotionKey (1 = aucune remise)"),
    "Montant du retour": ("Valeur financière des retours", "$", "Texte « $x,xxx.xx » à convertir"),
    "Quantité de retour": ("Nombre d'unités retournées", "unités", ""),
    "Montant des ventes": ("Chiffre d'affaires généré", "$", "= Prix unitaire × Quantité de vente − Montant de la remise"),
    "Quantité de vente": ("Nombre d'unités vendues", "unités", ""),
    "Coût total": ("Coût total des produits vendus", "$", "= Coût unitaire × Quantité de vente"),
    "Coût unitaire": ("Coût par unité", "$", ""),
    "Prix unitaire": ("Prix de vente par unité", "$", ""),
    "Montant de la remise": ("Valeur des remises appliquées", "$", "Remise sur une partie des unités seulement"),
    "Quantité de remise": ("Nombre d'unités avec remise", "unités", "≤ Quantité de vente"),
}


# ---------------------------------------------------------------------------
# 2. Profil de chaque variable (calculé sur toutes les ventes)
# ---------------------------------------------------------------------------
profil = []
for col in reference:
    s = ventes[col]

    # get() renvoie trois textes vides si la colonne n'a pas de description
    desc, unite, remarque = descriptions.get(col, ("", "", ""))

    # Le minimum, le maximum et la moyenne n'ont de sens que pour les nombres
    # (et le minimum / maximum pour les dates)
    numerique = pd.api.types.is_numeric_dtype(s)
    est_date = pd.api.types.is_datetime64_any_dtype(s)

    profil.append({
        "variable": col,
        "description": desc,
        "unité / rôle": unite,
        "type brut": "texte",                            # tout est du texte dans les fichiers
        "type converti": str(s.dtype),                   # type obtenu après conversion
        "valeurs manquantes": int(s.isna().sum()),
        "valeurs distinctes": int(s.nunique()),
        "minimum": s.min() if numerique or est_date else "",
        "maximum": s.max() if numerique or est_date else "",
        "moyenne": round(s.mean(), 2) if numerique else "",
        # exemple tel qu'il apparaît dans le premier fichier brut, avant conversion
        "exemple brut": next(iter(brut.values()))[col].iloc[0],
        "remarque": remarque,
    })

dictionnaire = pd.DataFrame(profil)
display(dictionnaire)


# ---------------------------------------------------------------------------
# 3. Liste des colonnes de chaque dimension
# ---------------------------------------------------------------------------
# Pour chaque table : sa taille, sa clé primaire et ses 12 premières colonnes
# (" …" indique qu'il y en a d'autres).
dimensions = pd.DataFrame([{
    "table": t,
    "lignes": len(d),
    "nb colonnes": d.shape[1],
    "clé primaire": d.columns[0],
    "colonnes": ", ".join(d.columns[:12]) + (" …" if d.shape[1] > 12 else ""),
} for t, d in dims.items()]) if dims else pd.DataFrame()


# ---------------------------------------------------------------------------
# 4. Enregistrer le dictionnaire dans un classeur Excel, un onglet par tableau
# ---------------------------------------------------------------------------
sortie = config.L1_L2 / "dictionnaire_donnees.xlsx"
with pd.ExcelWriter(sortie, engine="openpyxl") as xl:
    dictionnaire.to_excel(xl, sheet_name="Ventes", index=False)
    dimensions.to_excel(xl, sheet_name="Dimensions", index=False)
    profil_dims.to_excel(xl, sheet_name="Contrôle dimensions", index=False)
    liens.to_excel(xl, sheet_name="Liens dimensions", index=False)
    utilisation.to_excel(xl, sheet_name="Utilisation dimensions", index=False)
    # merge : une ligne par fichier, avec sa structure ET sa période
    structure.merge(periode, on="fichier").to_excel(xl, sheet_name="Fichiers", index=False)
    correspondance.to_excel(xl, sheet_name="Correspondance clés", index=False)
print("Dictionnaire enregistré :", sortie)

## 9. Périmètre : fichiers fournis et ventes complètes

**Ce qu'on a découvert.** Les six fichiers fournis par le professeur ne sont pas toutes les ventes de nos 56 boutiques en 2020 et 2021. Ce sont les lignes de la table `FactSales` de Contoso :

- limitées à **5 catégories de produits sur 8** (sans *Home Appliances*, *Cameras and camcorders* et *Games and Toys*) ;
- limitées aux **trimestres 1 à 3** de chaque année (sans les 4es trimestres).

À l'intérieur de ce périmètre, rien ne manque : chaque ligne fournie se retrouve à l'identique dans la base.

**Pourquoi on change de source.** `Extraction.py` extrait toutes les lignes de `FactSales` pour les mêmes 56 boutiques et les années entières, dans le même format que les fichiers du professeur (dossier `data/raw/contoso/Sales`). Trois raisons :

1. **Les charges.** Le plan stratégique de Contoso (`FactStrategyPlan`) donne, par boutique et par mois, les ventes réelles, le coût des ventes et les charges d'exploitation (personnel, loyers, publicité...). Ses ventes et son coût des ventes sont égaux aux totaux de `FactSales` complet **au cent près**. Ils ne sont pas comparables au périmètre partiel. Une rentabilité nette par boutique n'est donc possible qu'avec les ventes complètes.
2. **Le poids de ce qui manque.** Les trois catégories absentes représentent plus de la moitié du chiffre d'affaires. Les facteurs de vente (Q1) seraient mesurés sur une minorité de l'activité.
3. **La saisonnalité.** Sans les 4es trimestres, la fin d'année et la promotion des Fêtes (la plus forte, 20 %) sont tronquées, ce qui fausse Q2 et Q5.

Le périmètre du professeur reste reproductible : il suffit de mettre `SOURCE_VENTES = "professeur"` dans `src/config.py` et de relancer les notebooks.

La cellule suivante **prouve** ces affirmations à partir des deux dossiers de fichiers. La comparaison porte sur **2020 et 2021**, les deux années couvertes par les fichiers du professeur : l'année de référence 2019 en est écartée.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Charger les fichiers du professeur, pour les comparer aux ventes complètes
# ---------------------------------------------------------------------------
# La comparaison n'a de sens que si les notebooks lisent les ventes complètes
# ET que les fichiers du professeur sont dans data/raw/professeur.
fichiers_prof = etl.lister_fichiers_ventes(config.SALES_PROF)

if config.SOURCE_VENTES != "complet" or not fichiers_prof:
    print("Comparaison non exécutée : il faut SOURCE_VENTES = \"complet\" dans src/config.py")
    print("et les fichiers du professeur dans", config.SALES_PROF)
else:
    ventes_prof, _ = etl.charger_ventes(config.SALES_PROF)
    COL_CA = "Montant des ventes"
    # Les fichiers du professeur couvrent les deux années de l'énoncé : on compare sur ces années-là.
    # "ventes" (les trois années) n'est pas modifiée ; "ventes_enonce" en est la partie 2020-2021.
    ventes_enonce = ventes[ventes[config.COL_DATE].dt.year.isin(config.ANNEES_ATTENDUES)]

    # -----------------------------------------------------------------------
    # 2. Les lignes du professeur sont-elles toutes dans les ventes complètes, à l'identique ?
    # -----------------------------------------------------------------------
    # On rapproche les deux tables par IdVente, puis on compare les colonnes de mesure.
    mesures = ["Montant des ventes", "Quantité de vente", "Coût total", "Montant de la remise",
               "Quantité de remise", "Montant du retour", "Quantité de retour"]
    rapprochement = ventes_prof[[config.COL_VENTE] + mesures].merge(
        ventes_enonce[[config.COL_VENTE] + mesures], on=config.COL_VENTE, how="left",
        suffixes=("_prof", "_complet"), indicator=True)
    retrouvees = (rapprochement["_merge"] == "both").sum()
    # Plus grand écart observé sur une mesure, toutes lignes et toutes colonnes confondues
    ecart_max = max(float((rapprochement[f"{m}_prof"].astype(float) - rapprochement[f"{m}_complet"].astype(float)).abs().max())
                    for m in mesures)
    print(f"Lignes du professeur                         : {len(ventes_prof):,}".replace(",", " "))
    print(f"Retrouvées dans les ventes complètes (IdVente) : {retrouvees:,}".replace(",", " "))
    print(f"Plus grand écart sur une mesure               : {ecart_max:.2f}")

    # -----------------------------------------------------------------------
    # 3. Ce qui distingue les deux périmètres : la catégorie de produit et le trimestre
    # -----------------------------------------------------------------------
    # Chaîne produit -> sous-catégorie -> catégorie, à partir des dimensions chargées en section 5
    categorie_du_produit = (dims["DimProduct"][["ProductKey", "ProductSubcategoryKey"]]
                            .merge(dims["DimProductSubcategory"][["ProductSubcategoryKey", "ProductCategoryKey"]],
                                   on="ProductSubcategoryKey")
                            .merge(dims["DimProductCategory"][["ProductCategoryKey", "ProductCategoryName"]],
                                   on="ProductCategoryKey"))
    categorie_du_produit["ProductCategoryName"] = categorie_du_produit.ProductCategoryName.str.strip()
    # Dictionnaire IdProduit -> nom de la catégorie (les clés sont converties en entiers des deux côtés)
    nom_categorie = dict(zip(categorie_du_produit.ProductKey.astype(int), categorie_du_produit.ProductCategoryName))

    def resume(table, cle):
        """Lignes et chiffre d'affaires d'une table de ventes, par valeur de la série cle."""
        return table.groupby(cle).agg(lignes=(config.COL_VENTE, "size"), ca=(COL_CA, "sum"))

    # a) Par catégorie
    cat_complet = resume(ventes_enonce, ventes_enonce[config.COL_PRODUIT].astype(int).map(nom_categorie).rename("catégorie"))
    cat_prof = resume(ventes_prof, ventes_prof[config.COL_PRODUIT].astype(int).map(nom_categorie).rename("catégorie"))
    par_categorie = cat_complet.join(cat_prof, lsuffix="_complet", rsuffix="_prof").fillna(0)
    par_categorie["part_du_ca_complet"] = par_categorie.ca_complet.astype(float) / float(par_categorie.ca_complet.sum())
    par_categorie["dans_les_fichiers_du_prof"] = np.where(par_categorie.lignes_prof > 0, "oui", "NON")
    print("\nPar catégorie de produit :")
    display(par_categorie.sort_values("ca_complet", ascending=False)
            .style.format({"lignes_complet": "{:,.0f}", "lignes_prof": "{:,.0f}", "ca_complet": "{:,.0f}",
                           "ca_prof": "{:,.0f}", "part_du_ca_complet": "{:.1%}"}))

    # b) Par trimestre
    def trimestre(table):
        """Étiquette "2020 T1" de chaque ligne, d'après sa date."""
        d = table[config.COL_DATE]
        return (d.dt.year.astype(str) + " T" + d.dt.quarter.astype(str)).rename("trimestre")

    par_trimestre = (resume(ventes_enonce, trimestre(ventes_enonce))
                     .join(resume(ventes_prof, trimestre(ventes_prof)), lsuffix="_complet", rsuffix="_prof").fillna(0))
    print("Par trimestre :")
    display(par_trimestre.style.format("{:,.0f}"))

    # c) Contre-épreuve : si on applique aux ventes complètes le filtre "catégories du professeur
    #    et trimestres 1 à 3", retombe-t-on exactement sur le nombre de lignes du professeur ?
    categories_prof = set(cat_prof.index)
    filtre = (ventes_enonce[config.COL_PRODUIT].astype(int).map(nom_categorie).isin(categories_prof)
              & (ventes_enonce[config.COL_DATE].dt.quarter <= 3))
    print(f"Ventes complètes filtrées (catégories du professeur et trimestres 1 à 3) : {int(filtre.sum()):,} lignes".replace(",", " "),
          f"· fichiers du professeur : {len(ventes_prof):,} lignes".replace(",", " "))

    # -----------------------------------------------------------------------
    # 4. Rapprochement avec le plan stratégique de Contoso (FactStrategyPlan)
    # -----------------------------------------------------------------------
    # Scénario "Actual", comptes "Sale Revenue" et "Cost of Goods Sold", somme sur nos 56 boutiques,
    # lus dans le plan extrait par Extraction.py. Les trois années sont rapprochées, 2019 compris.
    plan_brut = pd.read_csv(config.PLAN / config.FICHIER_PLAN, encoding="utf-8-sig")
    reel = int(dims["DimScenario"].loc[dims["DimScenario"].ScenarioName == "Actual", "ScenarioKey"].iloc[0])
    cle_compte = dims["DimAccount"].set_index("AccountName").AccountKey
    plan_reel = plan_brut[plan_brut.ScenarioKey == reel].assign(annee=lambda d: d.Date.str[:4].astype(int))
    PLAN = {annee: {"ventes": float(g.loc[g.AccountKey == cle_compte["Sale Revenue"], "Montant"].sum()),
                    "cout": float(g.loc[g.AccountKey == cle_compte["Cost of Goods Sold"], "Montant"].sum())}
            for annee, g in plan_reel.groupby("annee")}
    lignes_plan = []
    for annee, attendu in PLAN.items():
        de_l_annee = ventes[ventes[config.COL_DATE].dt.year == annee]
        ca_complet = float(de_l_annee[COL_CA].sum())
        cout_complet = float(de_l_annee["Coût total"].sum())
        ca_prof = float(ventes_prof.loc[ventes_prof[config.COL_DATE].dt.year == annee, COL_CA].sum())
        lignes_plan.append({"année": annee, "ventes du plan": attendu["ventes"], "CA ventes complètes": ca_complet,
                            "écart ventes": ca_complet - attendu["ventes"],
                            "coût des ventes du plan": attendu["cout"], "coût total ventes complètes": cout_complet,
                            "écart coût": cout_complet - attendu["cout"],
                            "CA fichiers du professeur": ca_prof})
    print("\nRapprochement avec le plan stratégique :")
    display(pd.DataFrame(lignes_plan).set_index("année").style.format("{:,.2f}"))

    # -----------------------------------------------------------------------
    # 5. Bilan chiffré
    # -----------------------------------------------------------------------
    print(f"Fichiers du professeur : {len(ventes_prof):,} lignes, {float(ventes_prof[COL_CA].sum()):,.0f} $".replace(",", " "))
    print(f"Ventes complètes, 2020 et 2021 : {len(ventes_enonce):,} lignes, {float(ventes_enonce[COL_CA].sum()):,.0f} $".replace(",", " "))
    print(f"Part du chiffre d'affaires complet de 2020 et 2021 couverte par les fichiers du professeur : "
          f"{float(ventes_prof[COL_CA].sum()) / float(ventes_enonce[COL_CA].sum()):.1%}")
    print(f"Ventes complètes, avec l'année de référence {config.ANNEE_REFERENCE} : {len(ventes):,} lignes, {float(ventes[COL_CA].sum()):,.0f} $".replace(",", " "))

**Lecture**

- Toutes les lignes du professeur sont dans les ventes complètes, sans écart : les deux sources viennent bien de la même table.
- Trois catégories et les deux 4es trimestres sont entièrement absents des fichiers du professeur, et le filtre « 5 catégories, trimestres 1 à 3 » redonne exactement leur nombre de lignes. Ce n'est donc pas un échantillon tiré au hasard, c'est un **sous-périmètre**.
- Le chiffre d'affaires et le coût total des ventes complètes sont égaux aux montants du plan stratégique, **pour les trois années, au cent près**. Les charges du plan peuvent donc être rapprochées de nos ventes, boutique par boutique.

**Ce que cela change pour le projet** : tous les notebooks suivants travaillent sur les ventes complètes. Les constats du diagnostic qui dépendaient du périmètre (trimestres absents, quantité minimale par ligne, seuil des grandes quantités) sont établis dans `03_diagnostic`.

## 10. Les autres tables de faits : charges, objectifs, stock, ventes en ligne détaillées

Les ventes ne sont plus la seule table de faits du projet. Quatre autres tables décrivent les mêmes 56 boutiques sur les mêmes trois années. Avant de les utiliser, on fait pour chacune le même inventaire que pour les ventes :

| Contrôle | Pourquoi |
|---|---|
| Taille, période, rythme | Savoir ce qu'une ligne représente : un mois, une semaine, une commande |
| Valeurs manquantes | Repérer les colonnes incomplètes |
| Unicité de la clé | Une clé répétée fausserait toute somme |
| Boutiques et produits connus | Chaque clé doit exister dans nos dimensions |
| Recoupement avec les ventes | Deux tables indépendantes qui donnent le même total se valident l'une l'autre |

Ce notebook observe seulement. Le chargement de ces tables est fait dans `01_etl`, leur diagnostic détaillé dans `05b_charges` (plan stratégique) et `05c_stock_objectifs` (stock et objectifs).

In [ ]:
# ---------------------------------------------------------------------------
# 1. Lire les quatre tables telles qu'elles ont été extraites
# ---------------------------------------------------------------------------
plan_brut = pd.read_csv(config.PLAN / config.FICHIER_PLAN, encoding="utf-8-sig", parse_dates=["Date"])
objectifs_brut = pd.read_parquet(config.PLAN / config.FICHIER_OBJECTIFS)
stock_brut = pd.read_parquet(config.STOCK / config.FICHIER_STOCK)
en_ligne_brut = pd.concat([pd.read_parquet(f) for f in sorted(config.SALES_CONTOSO.glob(config.MOTIF_FICHIERS_EN_LIGNE))],
                          ignore_index=True)

AUTRES_FAITS = {
    # nom : (table, clé attendue unique, ce qu'une ligne représente)
    "FactStrategyPlan (charges)": (plan_brut, ["IdBoutique", "Date", "ScenarioKey", "AccountKey", "ProductCategoryKey"],
                                   "une boutique, un mois, un scénario, un compte, une catégorie"),
    "FactSalesQuota (objectifs)": (objectifs_brut, ["IdBoutique", "IdProduit", "Date", "ScenarioKey"],
                                   "une boutique, un produit, un mois, un scénario"),
    "FactInventory (stock)": (stock_brut, ["IdBoutique", "IdProduit", "Date"],
                              "une boutique, un produit, une semaine"),
    "FactOnlineSales (ventes en ligne détaillées)": (en_ligne_brut, ["IdVenteEnLigne"],
                                                     "une ligne de commande d'un client"),
}

boutiques_ventes = set(ventes[config.COL_BOUTIQUE].astype(int))
produits_connus = set(dims["DimProduct"].ProductKey.astype(int))

inventaire_faits = []
for nom, (table, cle, grain) in AUTRES_FAITS.items():
    dates = table["Date"]
    inventaire_faits.append({
        "table": nom,
        "une ligne représente": grain,
        "lignes": len(table),
        "colonnes": table.shape[1],
        "du": dates.min().date(), "au": dates.max().date(),
        "dates distinctes": dates.nunique(),
        "boutiques": table.IdBoutique.nunique(),
        "boutiques hors de nos ventes": len(set(table.IdBoutique.astype(int)) - boutiques_ventes),
        "produits": table.IdProduit.nunique() if "IdProduit" in table else np.nan,
        "produits inconnus de DimProduct": (len(set(table.IdProduit.astype(int)) - produits_connus) if "IdProduit" in table else np.nan),
        "cellules vides": int(table.isna().sum().sum()),
        "doublons sur la clé": int(table.duplicated(cle).sum()),
    })
inventaire_faits = pd.DataFrame(inventaire_faits).set_index("table")
display(inventaire_faits.T)

# ---------------------------------------------------------------------------
# 2. Lignes par année : l'activité décrite par chaque table évolue-t-elle ?
# ---------------------------------------------------------------------------
lignes_par_annee = pd.DataFrame({nom: table.groupby(table["Date"].dt.year).size() for nom, (table, _, _) in AUTRES_FAITS.items()})
lignes_par_annee.insert(0, "FactSales (ventes)", ventes[config.COL_DATE].dt.year.value_counts().sort_index())
lignes_par_annee.index.name = "année"
print("Lignes par année :")
display(lignes_par_annee.style.format("{:,.0f}"))

# ---------------------------------------------------------------------------
# 3. Recoupements : ces tables retrouvent-elles le total de nos ventes ?
# ---------------------------------------------------------------------------
def par_annee(table, colonne_date, colonne_montant):
    """Somme d'un montant par année. Les valeurs sont passées sans leur index : celui des ventes contient des répétitions."""
    return pd.Series(table[colonne_montant].astype(float).values).groupby(table[colonne_date].dt.year.values).sum()


scenario_reel = int(dims["DimScenario"].loc[dims["DimScenario"].ScenarioName == "Actual", "ScenarioKey"].iloc[0])
compte_ventes = int(dims["DimAccount"].loc[dims["DimAccount"].AccountName == "Sale Revenue", "AccountKey"].iloc[0])
boutique_en_ligne = sorted(int(b) for b in en_ligne_brut.IdBoutique.unique())
recoupement = pd.DataFrame({
    "ventes (FactSales)": par_annee(ventes, config.COL_DATE, "Montant des ventes"),
    "plan stratégique, ventes réelles": par_annee(plan_brut[(plan_brut.ScenarioKey == scenario_reel) & (plan_brut.AccountKey == compte_ventes)], "Date", "Montant"),
    "objectifs, scénario réel": par_annee(objectifs_brut[objectifs_brut.ScenarioKey == scenario_reel], "Date", "VentesObjectif"),
    "FactSales, boutique en ligne seule": par_annee(ventes[ventes[config.COL_BOUTIQUE].isin(boutique_en_ligne)], config.COL_DATE, "Montant des ventes"),
    "ventes en ligne détaillées": par_annee(en_ligne_brut, "Date", "MontantVentes"),
})
recoupement["écart plan"] = recoupement["plan stratégique, ventes réelles"] - recoupement["ventes (FactSales)"]
recoupement["écart objectifs"] = recoupement["objectifs, scénario réel"] - recoupement["ventes (FactSales)"]
recoupement["écart en ligne"] = recoupement["ventes en ligne détaillées"] - recoupement["FactSales, boutique en ligne seule"]
recoupement.index.name = "année"
print(f"Recoupement des montants de ventes, par année (boutique des ventes en ligne détaillées : {boutique_en_ligne}) :")
display(recoupement.T.style.format("{:,.2f}"))

# ---------------------------------------------------------------------------
# 4. Enregistrer cet inventaire dans le dictionnaire de données
# ---------------------------------------------------------------------------
with pd.ExcelWriter(sortie, engine="openpyxl", mode="a", if_sheet_exists="replace") as xl:
    inventaire_faits.reset_index().to_excel(xl, sheet_name="Autres tables de faits", index=False)
    lignes_par_annee.reset_index().to_excel(xl, sheet_name="Lignes par année", index=False)
    recoupement.reset_index().to_excel(xl, sheet_name="Recoupement des ventes", index=False)
    for nom, (table, _, _) in AUTRES_FAITS.items():
        colonnes_table = pd.DataFrame({"colonne": table.columns, "type": [str(t) for t in table.dtypes],
                                       "valeurs manquantes": table.isna().sum().values,
                                       "valeurs distinctes": [table[c].nunique() for c in table.columns],
                                       "minimum": [table[c].min() for c in table.columns],
                                       "maximum": [table[c].max() for c in table.columns]})
        colonnes_table.to_excel(xl, sheet_name=nom.split(" ")[0][:31], index=False)
print("Inventaire ajouté au dictionnaire :", sortie)

**Lecture**

- **Aucune cellule vide et aucun doublon de clé** dans les quatre tables. Toutes leurs boutiques sont les nôtres, et tous leurs produits existent dans `DimProduct`.
- **Deux recoupements exacts.** Les ventes réelles du plan stratégique et le scénario réel des objectifs sont égaux aux ventes de `FactSales`, pour les trois années, au cent près. Trois tables extraites séparément donnent le même total : c'est la meilleure preuve que nos données sont complètes.
- **Un recoupement qui échoue.** Les ventes en ligne détaillées portent bien sur notre boutique en ligne, mais leur total ne correspond pas à celui de cette boutique dans `FactSales`, et l'écart change de sens selon l'année. Ce sont deux sources qui ne décrivent pas les mêmes ventes. Conséquence : cette table peut servir à décrire des clients et des commandes, **pas à corriger ou à compléter nos montants**. Le diagnostic (`03_diagnostic`) revient sur ce point.
- **Le nombre de lignes baisse partout d'année en année**, sauf dans les ventes en ligne détaillées. Le stock en particulier perd les deux tiers de ses lignes en deux ans, alors que le nombre de produits augmente : il y a donc de moins en moins de couples boutique et produit suivis. C'est le point de départ de `06b_assortiment`.
- **Les rythmes diffèrent** : les ventes sont quotidiennes, le stock hebdomadaire, le plan et les objectifs mensuels. Toute comparaison entre ces tables se fait donc au mois.

## 11. Synthèse pour la présentation et le rapport

In [ ]:
# ---------------------------------------------------------------------------
# Tableau des chiffres clés : (indicateur, valeur)
# ---------------------------------------------------------------------------
lignes_synthese = [
    ("Fichiers de ventes", f"{len(fichiers)} (trimestres attendus : {len(attendus)})"),
    # join() relie les trimestres absents par une virgule ; "aucun" si la liste est vide
    ("Trimestres absents", ", ".join(f"{a} T{t}" for a, t in manquants_reels) or "aucun"),
    ("Ventes (lignes), trois années", f"{len(ventes):,}".replace(",", " ")),
    ("dont années de l'énoncé (2020 et 2021)", f"{int(ventes[config.COL_DATE].dt.year.isin(config.ANNEES_ATTENDUES).sum()):,}".replace(",", " ")),
    ("Période couverte", f"{ventes[config.COL_DATE].min().date()} au {ventes[config.COL_DATE].max().date()}"),
    # ":,.0f" : séparateur de milliers, aucune décimale
    ("Chiffre d'affaires total, trois années", f"{ventes['Montant des ventes'].sum():,.0f} $".replace(",", " ")),
    ("dont années de l'énoncé (2020 et 2021)",
     f"{ventes.loc[ventes[config.COL_DATE].dt.year.isin(config.ANNEES_ATTENDUES), 'Montant des ventes'].sum():,.0f} $".replace(",", " ")),
    ("Boutiques / produits / promotions",
     f"{ventes[config.COL_BOUTIQUE].nunique()} / {ventes[config.COL_PRODUIT].nunique()} / {ventes[config.COL_PROMOTION].nunique()}"),
    ("IdVente en double", f"{doublons_total.sum():,}".replace(",", " ")),
]

# Une ligne par clé contrôlée à la section 5
lignes_synthese += [
    (f"Correspondance {r['clé (nos ventes)']}", f"{r['% des lignes retrouvées']} %")
    for _, r in correspondance.iterrows()
]

# Contrôles des dimensions (section 6)
if len(profil_dims):
    lignes_synthese += [
        ("Dimensions contrôlées", f"{len(profil_dims)} tables"),
        # all() : True seulement si TOUTES les tables ont une clé unique
        ("Clés primaires uniques", "oui" if profil_dims["clé unique"].all() else "non : voir section 6"),
        ("Orphelins entre dimensions", str(int(liens.orphelins.sum())) if len(liens) else "non vérifié"),
    ]

# Autres tables de faits (section 10)
lignes_synthese += [(f"{nom} (lignes)", f"{len(table):,}".replace(",", " ")) for nom, (table, _, _) in AUTRES_FAITS.items()]
lignes_synthese += [("Plan et objectifs égaux aux ventes", "oui, au cent près" if (recoupement[["écart plan", "écart objectifs"]].abs().max().max() < 0.01) else "non : voir section 10")]

synthese = pd.DataFrame(lignes_synthese, columns=["indicateur", "valeur"])
display(synthese)